In [8]:
import tacoreader
import rasterio as rio
import numpy as np

In [9]:
dataset = tacoreader.load([ r"G:\Meu Drive\taco_CloudSen12\l1c/cloudsen12-l1c.0000.part.taco",
                            r"G:\Meu Drive\taco_CloudSen12\l1c/cloudsen12-l1c.0001.part.taco",
                            r"G:\Meu Drive\taco_CloudSen12\l1c/cloudsen12-l1c.0002.part.taco",
                            r"G:\Meu Drive\taco_CloudSen12\l1c/cloudsen12-l1c.0003.part.taco",
                            r"G:\Meu Drive\taco_CloudSen12\l1c/cloudsen12-l1c.0004.part.taco",
                            ])
df = dataset[(dataset["label_type"] == "high") & (dataset["real_proj_shape"] == 509)]

train_dataset = df[df["tortilla:data_split"] == "train"]

In [11]:
limpo = 0
nuvem = 0
nuvem_fina = 0
sombra = 0

for i in range(len(train_dataset)):
    sample = train_dataset.read(i)
    s2l1c: str = sample.read(0)
    target: str = sample.read(1)
    
    # Open the files and load data
    with rio.open(target) as dst:
        target_data: np.ndarray = dst.read().astype(np.int64)

        limpo += np.sum(target_data == 0)
        nuvem += np.sum(target_data == 1)
        nuvem_fina += np.sum(target_data == 2)
        sombra += np.sum(target_data == 3)

print("Total pixels:", limpo + nuvem + nuvem_fina + sombra)
print("Clean pixels:", limpo)
print("Cloud pixels:", nuvem)
print("Thin cloud pixels:", nuvem_fina)
print("Shadow pixels:", sombra)

# Calculate weights for cross-entropy loss

total = limpo + nuvem + nuvem_fina + sombra
weights = {
    "limpo": total / limpo,
    "nuvem": total / nuvem,
    "nuvem_fina": total / nuvem_fina,
    "sombra": total / sombra
}

print("Class weights:", weights)

Total pixels: 2225602560
Clean pixels: 1223366733
Cloud pixels: 590595150
Thin cloud pixels: 214564224
Shadow pixels: 197076453
Class weights: {'limpo': np.float64(1.8192439764503552), 'nuvem': np.float64(3.76840642866776), 'nuvem_fina': np.float64(10.37266380438148), 'sombra': np.float64(11.293092229542005)}


In [12]:
total = limpo + nuvem + nuvem_fina + sombra
weights = {
    "limpo": limpo / limpo,
    "nuvem": limpo / nuvem,
    "nuvem_fina": limpo / nuvem_fina,
    "sombra": limpo / sombra
}

print("Class weights:", weights)

Class weights: {'limpo': np.float64(1.0), 'nuvem': np.float64(2.071413442863525), 'nuvem_fina': np.float64(5.7016342715176975), 'sombra': np.float64(6.207574341720063)}
